# Portfolio Optimization — Data Pipeline
### BA870/AC820 Spring 2026 | Team: Akbar Wibowo, Danish Azmi, Samuel Buelvas

**30-Stock / 5-Industry Black-Litterman Pipeline**

This notebook is the single source for all data loading, merging, feature engineering, and Black-Litterman input computation. All three team members work from this notebook:

| Member | Primary Role | Key Sections |
|--------|-------------|--------------|
| **A (Data)** | Owns this notebook | Sections 1–6 |
| **B (ML)** | Imports training data | Section 7A |
| **C (Optimization)** | Imports BL inputs | Section 7B |

**Run order:**
1. Install dependencies (Section 0)
2. Verify WRDS access (Section 1)
3. Pull all data (Sections 2–4)
4. Merge & engineer features (Sections 5–6)
5. Export for teammates (Section 7)

---

## Section 0 — Setup & Configuration

Install all dependencies first. Run this cell once.

In [1]:
# Run once — install all project dependencies
# If running on Google Colab, these may already be installed.

!pip install -q wrds yfinance pandas-datareader fredapi \
    scikit-learn xgboost scipy plotly streamlit \
    pyarrow statsmodels arch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.3/981.3 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 36.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 40.2 MB/s eta 0:00:00


In [2]:
import os
import warnings
import numpy as np
import pandas as pd
from datetime import datetime, timedelta

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:.4f}".format)

print("Setup complete ✓")

Setup complete ✓


In [3]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


### Stock Universe Configuration

30 stocks across 5 GICS sectors, 6 per sector. The **PERMNOs below are placeholders** — run Section 1 to look up the correct values and paste them back here.

In [4]:
# =============================================================================
# STOCK UNIVERSE — 30 Stocks, 5 GICS Sectors, 6 per Sector
# =============================================================================
# ⚠️  PERMNOs are PLACEHOLDERS. Run Section 1 to get verified values.
#     CRSP uses PERMNO (not ticker) as the permanent identifier.

UNIVERSE = {
    "Information Technology": {
        "gics_code": 45,
        "stocks": {
            "AAPL":  14593,   # Apple
            "MSFT":  10107,   # Microsoft
            "NVDA":  93436,   # NVIDIA
            "AVGO":  15966,   # Broadcom
            "ORCL":  10104,   # Oracle
            "CRM":   89399,   # Salesforce
        },
    },
    "Health Care": {
        "gics_code": 35,
        "stocks": {
            "LLY":   48598,   # Eli Lilly
            "UNH":   92655,   # UnitedHealth
            "JNJ":   22111,   # Johnson & Johnson
            "ABBV":  92611,   # AbbVie
            "MRK":   22752,   # Merck
            "TMO":   85606,   # Thermo Fisher
        },
    },
    "Financials": {
        "gics_code": 40,
        "stocks": {
            "BRK-B": 83443,   # Berkshire Hathaway B
            "JPM":   47896,   # JPMorgan Chase
            "V":     93436,   # Visa       — VERIFY PERMNO
            "MA":    89399,   # Mastercard — VERIFY PERMNO
            "BAC":   59408,   # Bank of America
            "GS":    86868,   # Goldman Sachs
        },
    },
    "Consumer Discretionary": {
        "gics_code": 25,
        "stocks": {
            "AMZN":  84788,   # Amazon
            "TSLA":  93436,   # Tesla — VERIFY PERMNO
            "HD":    66181,   # Home Depot
            "MCD":   43449,   # McDonald's
            "NKE":   57665,   # Nike
            "SBUX":  82542,   # Starbucks
        },
    },
    "Energy": {
        "gics_code": 10,
        "stocks": {
            "XOM":   11850,   # Exxon Mobil
            "CVX":   14541,   # Chevron
            "COP":   87813,   # ConocoPhillips
            "EOG":   82373,   # EOG Resources
            "SLB":   26403,   # Schlumberger
            "PSX":   92449,   # Phillips 66
        },
    },
}

# --- Build flat lookup dictionaries ---
TICKER_TO_PERMNO = {}
PERMNO_TO_TICKER = {}
TICKER_TO_SECTOR = {}

for sector, info in UNIVERSE.items():
    for ticker, permno in info["stocks"].items():
        TICKER_TO_PERMNO[ticker] = permno
        PERMNO_TO_TICKER[permno] = ticker
        TICKER_TO_SECTOR[ticker] = sector

ALL_PERMNOS = list(TICKER_TO_PERMNO.values())
ALL_TICKERS = list(TICKER_TO_PERMNO.keys())

print(f"Universe: {len(ALL_TICKERS)} stocks across {len(UNIVERSE)} sectors")
for sector, info in UNIVERSE.items():
    tickers = list(info["stocks"].keys())
    print(f"  {sector}: {', '.join(tickers)}")

Universe: 30 stocks across 5 sectors
  Information Technology: AAPL, MSFT, NVDA, AVGO, ORCL, CRM
  Health Care: LLY, UNH, JNJ, ABBV, MRK, TMO
  Financials: BRK-B, JPM, V, MA, BAC, GS
  Consumer Discretionary: AMZN, TSLA, HD, MCD, NKE, SBUX
  Energy: XOM, CVX, COP, EOG, SLB, PSX


In [5]:
# =============================================================================
# PIPELINE PARAMETERS
# =============================================================================

# Date range — pull extra history for rolling window warm-up
TRAIN_START = "2016-01-01"    # warm-up buffer for 252-day rolling features
DATA_END    = "2025-12-31"    # adjust to current date

# Feature engineering windows (trading days)
ROLLING_WINDOWS = {
    "vol_short":   21,   # ~1 month
    "vol_long":    63,   # ~3 months
    "mom_3m":      63,
    "mom_6m":     126,
    "mom_12m":    252,
    "beta_window": 252,  # 1 year for rolling beta
}

# ML target
FORWARD_RETURN_HORIZON = 21   # predict 21-day forward return (~1 month)

# Black-Litterman
BL_RISK_AVERSION = 2.5       # δ — market risk aversion coefficient
BL_TAU = 0.05                # uncertainty scaling on equilibrium

# Compustat reporting lag (days) — prevents look-ahead bias
REPORTING_LAG_DAYS = 60

print("Parameters configured ✓")

Parameters configured ✓


---
## Section 1 — Verify WRDS Connection & Look Up PERMNOs

**Run this section first.** It tests your WRDS connection, finds the correct PERMNOs for all 30 tickers, and checks data availability. After running, **copy the verified PERMNOs back into the UNIVERSE config above.**

In [6]:
# =============================================================================
# 1A. CONNECT TO WRDS
# =============================================================================
# On first run, you'll be prompted for your WRDS password.
# After that, it's saved in ~/.pgpass and you won't be asked again.

import wrds

# Try with explicit SSL and retry
conn = wrds.Connection(
    wrds_username='',    # ← check this is your real WRDS username
)

# Set your WRDS username here or via environment variable
WRDS_USERNAME = os.environ.get("WRDS_USERNAME", "your_username_here")

conn = wrds.Connection(wrds_username=WRDS_USERNAME)
print("Connected to WRDS ✓")

Enter your WRDS username [root]:akbarwibowo
Enter your password:··········
WRDS recommends setting up a .pgpass file.
Create .pgpass file now [y/n]?: y
Created .pgpass file successfully.
You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Done
Enter your WRDS username [your_username_here]:akbarwibowo
Enter your password:··········
WRDS recommends setting up a .pgpass file.
Create .pgpass file now [y/n]?: y
Created .pgpass file successfully.
You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Done
Connected to WRDS ✓


In [7]:
# Run this if first time running on your account
import shutil
shutil.copy(os.path.expanduser("~/.pgpass"), "/content/drive/MyDrive/")

'/content/drive/MyDrive/.pgpass'

In [8]:
# For future re-runs USE THIS
#import shutil, os
#shutil.copy("/content/drive/MyDrive/BA870_Project/.pgpass", os.path.expanduser("~/.pgpass"))
#os.chmod(os.path.expanduser("~/.pgpass"), 0o600)

In [8]:
# Look up the actual CRSP tickers for Berkshire and Schlumberger
fix_query = """
    SELECT ticker, permno, namedt, nameenddt, shrcd, exchcd
    FROM crsp.stocknames
    WHERE (ticker LIKE 'BRK%%' OR ticker LIKE 'SLB%%')
        AND nameenddt >= '2024-01-01'
        AND shrcd IN (10, 11)
    ORDER BY ticker, nameenddt DESC
"""
conn.raw_sql(fix_query)

,ticker,permno,namedt,nameenddt,shrcd,exchcd
0,BRK,17778,2021-04-05,2024-12-31,11,1
1,BRK,83443,2021-04-05,2024-12-31,11,1
2,BRKH,22635,2022-01-31,2024-12-17,11,3
3,BRKL,85860,2002-07-10,2024-12-31,11,3
4,BRKR,88504,2008-02-27,2024-12-31,11,3


In [9]:
slb_query = """
    SELECT ticker, permno, comnam, namedt, nameenddt
    FROM crsp.stocknames
    WHERE comnam LIKE '%%SCHLUMBERGER%%' OR comnam LIKE '%%SLB%%'
    ORDER BY nameenddt DESC
    LIMIT 10
"""
conn.raw_sql(slb_query)

,ticker,permno,comnam,namedt,nameenddt
0,SLB,14277,SCHLUMBERGER LTD,2020-04-06,2024-12-31
1,SLB,14277,SCHLUMBERGER LTD,2001-08-24,2020-04-05
2,SLB,14277,SCHLUMBERGER LTD,1968-01-02,2001-08-23
3,SLB,14277,SCHLUMBERGER LTD,1962-07-02,1968-01-01
4,<NA>,14277,SCHLUMBERGER LTD,1962-02-02,1962-07-01


In [10]:
# =============================================================================
# 1B. LOOK UP PERMNOs FOR ALL 30 TICKERS
# =============================================================================
# CRSP's stocknames table maps tickers → PERMNOs.
# We take the most recent mapping for each ticker.

ticker_str = "', '".join(ALL_TICKERS)

permno_query = f"""
    SELECT DISTINCT ON (ticker)
        ticker,
        permno,
        namedt,
        nameenddt,
        shrcd,
        exchcd
    FROM crsp.stocknames
    WHERE ticker IN ('{ticker_str}')
        AND nameenddt >= '2024-01-01'
        AND shrcd IN (10, 11)
        AND exchcd IN (1, 2, 3)
    ORDER BY ticker, nameenddt DESC
"""

permno_results = conn.raw_sql(permno_query)

# Display results
print(f"Found {len(permno_results)}/{len(ALL_TICKERS)} tickers in CRSP:\n")
print(f"{'Ticker':<8} {'PERMNO':<10} {'Sector':<28} {'Exchange'}")
print(f"{'-'*8} {'-'*10} {'-'*28} {'-'*10}")

verified_permnos = {}
for _, row in permno_results.iterrows():
    ticker = row["ticker"]
    permno = int(row["permno"])
    verified_permnos[ticker] = permno
    sector = TICKER_TO_SECTOR.get(ticker, "Unknown")
    exch = {1: "NYSE", 2: "AMEX", 3: "NASDAQ"}.get(row["exchcd"], "?")
    print(f"{ticker:<8} {permno:<10} {sector:<28} {exch}")

# Flag missing tickers
missing = set(ALL_TICKERS) - set(verified_permnos.keys())
if missing:
    print(f"\n⚠️  Missing tickers: {missing}")
    print("   Common issues: BRK-B might be BRK.B, or the stock may have a different ticker in CRSP")
else:
    print(f"\n✅ All {len(ALL_TICKERS)} tickers found")

print("\n📋 Copy these PERMNOs into the UNIVERSE config in Section 0 above.")

Found 28/30 tickers in CRSP:

Ticker   PERMNO     Sector                       Exchange
-------- ---------- ---------------------------- ----------
AAPL     14593      Information Technology       NASDAQ
ABBV     13721      Health Care                  NYSE
AMZN     84788      Consumer Discretionary       NASDAQ
AVGO     93002      Information Technology       NASDAQ
BAC      59408      Financials                   NYSE
COP      13928      Energy                       NYSE
CRM      90215      Information Technology       NYSE
CVX      14541      Energy                       NYSE
EOG      75825      Energy                       NYSE
GS       86868      Financials                   NYSE
HD       66181      Consumer Discretionary       NYSE
JNJ      22111      Health Care                  NYSE
JPM      47896      Financials                   NYSE
LLY      50876      Health Care                  NYSE
MA       91233      Financials                   NYSE
MCD      43449      Consumer Discret

In [11]:
# Add the two that were missing
verified_permnos["BRK-B"] = 83443
verified_permnos["SLB"] = 14277

# Verify you have all 30
print(f"Total stocks: {len(verified_permnos)}/30")
for ticker, permno in sorted(verified_permnos.items()):
    print(f"  {ticker:<8} {permno}")

Total stocks: 30/30
  AAPL     14593
  ABBV     13721
  AMZN     84788
  AVGO     93002
  BAC      59408
  BRK-B    83443
  COP      13928
  CRM      90215
  CVX      14541
  EOG      75825
  GS       86868
  HD       66181
  JNJ      22111
  JPM      47896
  LLY      50876
  MA       91233
  MCD      43449
  MRK      22752
  MSFT     10107
  NKE      57665
  NVDA     86580
  ORCL     10104
  PSX      13356
  SBUX     77702
  SLB      14277
  TMO      62092
  TSLA     93436
  UNH      92655
  V        92611
  XOM      11850


In [12]:
# =============================================================================
# 1C. CHECK CRSP DATA AVAILABILITY
# =============================================================================

if verified_permnos:
    sample = list(verified_permnos.values())[:6]
    permno_str = ", ".join(str(p) for p in sample)

    avail_query = f"""
        SELECT permno,
               MIN(date) AS first_date,
               MAX(date) AS last_date,
               COUNT(*)  AS n_trading_days
        FROM crsp.dsf
        WHERE permno IN ({permno_str})
            AND date BETWEEN '2016-01-01' AND '2025-12-31'
            AND ret IS NOT NULL
        GROUP BY permno
        ORDER BY permno
    """
    availability = conn.raw_sql(avail_query, date_cols=["first_date", "last_date"])

    print("Data availability (sample of 6 stocks):\n")
    print(f"{'PERMNO':<10} {'Ticker':<8} {'First':<14} {'Last':<14} {'Trading Days'}")
    print(f"{'-'*10} {'-'*8} {'-'*14} {'-'*14} {'-'*12}")

    rev_map = {v: k for k, v in verified_permnos.items()}
    for _, row in availability.iterrows():
        p = int(row["permno"])
        t = rev_map.get(p, "???")
        print(f"{p:<10} {t:<8} {str(row['first_date'].date()):<14} "
              f"{str(row['last_date'].date()):<14} {int(row['n_trading_days'])}")

Data availability (sample of 6 stocks):

PERMNO     Ticker   First          Last           Trading Days
---------- -------- -------------- -------------- ------------
13721      ABBV     2016-01-04     2024-12-31     2264
13928      COP      2016-01-04     2024-12-31     2264
14593      AAPL     2016-01-04     2024-12-31     2264
59408      BAC      2016-01-04     2024-12-31     2264
84788      AMZN     2016-01-04     2024-12-31     2264
93002      AVGO     2016-01-04     2024-12-31     2264


In [13]:
# =============================================================================
# 1D. CHECK COMPUSTAT CCM LINK
# =============================================================================

if verified_permnos:
    test_permno = list(verified_permnos.values())[0]
    test_ticker = PERMNO_TO_TICKER.get(test_permno, list(verified_permnos.keys())[0])

    ccm_query = f"""
        SELECT b.lpermno AS permno, a.gvkey, a.datadate, a.epspxq, a.prccq
        FROM comp.fundq AS a
        INNER JOIN crsp.ccmxpf_lnkhist AS b
            ON a.gvkey = b.gvkey
            AND b.linktype IN ('LU', 'LC')
            AND b.linkprim IN ('P', 'C')
            AND a.datadate BETWEEN b.linkdt AND COALESCE(b.linkenddt, CURRENT_DATE)
        WHERE b.lpermno = {test_permno}
            AND a.datadate >= '2023-01-01'
            AND a.datafmt = 'STD'
            AND a.indfmt  = 'INDL'
            AND a.consol  = 'C'
            AND a.popsrc  = 'D'
        ORDER BY a.datadate DESC
        LIMIT 8
    """

    ccm_test = conn.raw_sql(ccm_query, date_cols=["datadate"])

    if len(ccm_test) > 0:
        print(f"✅ Compustat CCM link working. Sample: {test_ticker} (PERMNO {test_permno})\n")
        print(ccm_test.to_string(index=False))
    else:
        print(f"⚠️  CCM link returned no data for {test_ticker}. Check Compustat access on WRDS.")

✅ Compustat CCM link working. Sample: AAPL (PERMNO 14593)

    permno  gvkey   datadate  epspxq    prccq
14593.0000 001690 2025-12-31  2.8500 271.8600
14593.0000 001690 2025-09-30  1.8500 254.6300
14593.0000 001690 2025-06-30  1.5700 205.1700
14593.0000 001690 2025-03-31  1.6500 222.1300
14593.0000 001690 2024-12-31  2.4100 250.4200
14593.0000 001690 2024-09-30  0.9700 233.0000
14593.0000 001690 2024-06-30  1.4000 210.6200
14593.0000 001690 2024-03-31  1.5300 171.4800


---
## Section 2 — Load Raw Data from All Sources

Four data sources, loaded in parallel-friendly order:
1. **CRSP Daily** (primary — prices, returns, volume, market cap)
2. **Compustat Quarterly** (P/E ratio via CCM link)
3. **Fama-French Factors** (market beta, SMB, HML, risk-free rate)
4. **FRED Macro** (VIX, term spread)

In [14]:
# =============================================================================
# 2A. CRSP DAILY STOCK FILE
# =============================================================================

def load_crsp_daily(conn, permnos=None, start_date=TRAIN_START, end_date=DATA_END):
    """
    Pull daily stock data from CRSP.

    Returns: DataFrame with permno, date, ticker, sector, ret, prc, vol, shrout, mkt_cap
    """
    if permnos is None:
        permnos = ALL_PERMNOS

    permno_str = ", ".join(str(p) for p in permnos)

    query = f"""
        SELECT
            a.permno,
            a.date,
            a.ret,
            ABS(a.prc)              AS prc,
            a.vol,
            a.shrout,
            ABS(a.prc) * a.shrout   AS mkt_cap
        FROM crsp.dsf AS a
        WHERE
            a.permno IN ({permno_str})
            AND a.date BETWEEN '{start_date}' AND '{end_date}'
            AND a.ret IS NOT NULL
        ORDER BY a.permno, a.date
    """

    print("[CRSP] Pulling daily data from WRDS...")
    df = conn.raw_sql(query, date_cols=["date"])

    # Map identifiers
    df["ticker"] = df["permno"].map(PERMNO_TO_TICKER)
    df["sector"] = df["ticker"].map(TICKER_TO_SECTOR)

    # Clean
    df = df.dropna(subset=["ret", "prc"])
    df["mkt_cap"] = df["mkt_cap"].abs()
    df = df.sort_values(["permno", "date"]).reset_index(drop=True)

    print(f"[CRSP] ✓ {len(df):,} rows, {df['permno'].nunique()} stocks, "
          f"{df['date'].min().date()} to {df['date'].max().date()}")
    return df

# --- Run it ---
crsp_df = load_crsp_daily(conn)
crsp_df.head()

[CRSP] Pulling daily data from WRDS...
[CRSP] ✓ 46,865 rows, 22 stocks, 2016-01-04 to 2024-12-31


,permno,date,ret,prc,vol,shrout,mkt_cap,ticker,sector
0,10104,2016-01-04,-0.0172,35.7500,18784386.0000,4201220.0000,150193615.0000,ORCL,Information Technology
1,10104,2016-01-05,-0.0031,35.6400,25340678.0000,4201220.0000,149731480.8000,ORCL,Information Technology
2,10104,2016-01-06,0.0051,35.8200,18165714.0000,4201220.0000,150487700.4000,ORCL,Information Technology
3,10104,2016-01-07,-0.0218,35.0400,22591403.0000,4201220.0000,147210748.8000,ORCL,Information Technology
4,10104,2016-01-08,-0.0111,34.6500,21962204.0000,4201220.0000,145572273.0000,ORCL,Information Technology


In [15]:
# =============================================================================
# 2B. COMPUSTAT QUARTERLY FUNDAMENTALS (P/E Ratio)
# =============================================================================

def load_compustat_quarterly(conn, permnos=None, start_date=TRAIN_START, end_date=DATA_END):
    """
    Pull quarterly EPS from Compustat via CCM link and compute trailing P/E.
    Applies a reporting lag to prevent look-ahead bias.

    Returns: DataFrame with permno, datadate, available_date, trailing_eps, pe_ratio
    """
    if permnos is None:
        permnos = ALL_PERMNOS

    permno_str = ", ".join(str(p) for p in permnos)

    query = f"""
        SELECT
            b.lpermno           AS permno,
            a.datadate,
            a.epspxq,
            a.prccq
        FROM comp.fundq AS a
        INNER JOIN crsp.ccmxpf_lnkhist AS b
            ON a.gvkey = b.gvkey
            AND b.linktype  IN ('LU', 'LC')
            AND b.linkprim  IN ('P', 'C')
            AND a.datadate BETWEEN b.linkdt AND COALESCE(b.linkenddt, CURRENT_DATE)
        WHERE
            b.lpermno IN ({permno_str})
            AND a.datadate BETWEEN '{start_date}' AND '{end_date}'
            AND a.datafmt = 'STD'
            AND a.indfmt  = 'INDL'
            AND a.consol  = 'C'
            AND a.popsrc  = 'D'
        ORDER BY b.lpermno, a.datadate
    """

    print("[Compustat] Pulling quarterly fundamentals via CCM link...")
    df = conn.raw_sql(query, date_cols=["datadate"])

    if df.empty:
        print("[Compustat] ⚠️  No data returned.")
        return pd.DataFrame(columns=["permno", "datadate", "available_date",
                                     "trailing_eps", "pe_ratio"])

    df = df.sort_values(["permno", "datadate"]).reset_index(drop=True)

    # Trailing 12-month EPS = sum of last 4 quarters
    df["trailing_eps"] = (
        df.groupby("permno")["epspxq"]
          .transform(lambda x: x.rolling(4, min_periods=4).sum())
    )

    # P/E ratio (only where EPS > 0 to avoid meaningless values)
    df["pe_ratio"] = np.where(
        df["trailing_eps"] > 0,
        df["prccq"] / df["trailing_eps"],
        np.nan,
    )

    # Reporting lag: data isn't available until ~60 days after quarter end
    df["available_date"] = df["datadate"] + pd.Timedelta(days=REPORTING_LAG_DAYS)

    df = df[["permno", "datadate", "available_date", "trailing_eps", "pe_ratio"]]
    print(f"[Compustat] ✓ {len(df):,} quarterly observations")
    return df

# --- Run it ---
compustat_df = load_compustat_quarterly(conn)
compustat_df.head()

[Compustat] Pulling quarterly fundamentals via CCM link...
[Compustat] ✓ 821 quarterly observations


,permno,datadate,available_date,trailing_eps,pe_ratio
0,10104.0000,2016-02-29,2016-04-29,NaN,NaN
1,10104.0000,2016-05-31,2016-07-30,NaN,NaN
2,10104.0000,2016-08-31,2016-10-30,NaN,NaN
3,10104.0000,2016-11-30,2017-01-29,2.1300,18.8685
4,10104.0000,2017-02-28,2017-04-29,2.1700,19.6267


In [17]:
# =============================================================================
# 2C. FAMA-FRENCH FACTORS (free — no WRDS needed)
# =============================================================================

def load_ff_factors(start_date=TRAIN_START, end_date=DATA_END):
    """
    Load Fama-French 3-factor daily returns from Kenneth French Data Library.
    Falls back to direct CSV download if pandas_datareader fails.

    Returns: DataFrame with date, mkt_rf, smb, hml, rf (all in decimal)
    """
    try:
        import pandas_datareader.data as web
        print("[FF] Downloading via pandas_datareader...")
        ff_raw = web.DataReader(
            "F-F_Research_Data_Factors_daily", "famafrench",
            start=start_date, end=end_date,
        )[0]
    except Exception as e:
        print(f"[FF] pandas_datareader failed ({e}), trying direct CSV...")
        import io, zipfile, urllib.request
        url = ("https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/"
               "ftp/F-F_Research_Data_Factors_daily_CSV.zip")
        resp = urllib.request.urlopen(url)
        z = zipfile.ZipFile(io.BytesIO(resp.read()))
        csv_name = z.namelist()[0]
        with z.open(csv_name) as f:
            lines = f.read().decode("utf-8").splitlines()
        data_lines = [l for l in lines
                      if len(l.strip().split(",")) >= 5
                      and l.strip().split(",")[0].strip().isdigit()
                      and len(l.strip().split(",")[0].strip()) == 8]
        csv_text = "date,Mkt-RF,SMB,HML,RF\n" + "\n".join(data_lines)
        ff_raw = pd.read_csv(io.StringIO(csv_text))
        ff_raw["date"] = pd.to_datetime(ff_raw["date"], format="%Y%m%d")
        ff_raw = ff_raw[(ff_raw["date"] >= start_date) & (ff_raw["date"] <= end_date)]
        ff_raw = ff_raw.set_index("date")

    # Convert from percentage to decimal
    ff = (ff_raw / 100.0).reset_index()
    ff.columns = [c.strip().lower().replace("-", "_") for c in ff.columns]
    ff["date"] = pd.to_datetime(ff["date"])
    ff = ff[(ff["date"] >= start_date) & (ff["date"] <= end_date)]

    print(f"[FF] ✓ {len(ff):,} daily observations, {ff['date'].min().date()} to {ff['date'].max().date()}")
    return ff[["date", "mkt_rf", "smb", "hml", "rf"]]

# --- Run it ---
ff_df = load_ff_factors()
ff_df.tail()

[FF] Downloading via pandas_datareader...
[FF] ✓ 2,514 daily observations, 2016-01-04 to 2025-12-31


,date,mkt_rf,smb,hml,rf
2509,2025-12-24,0.0029,0.0003,0.0001,0.0002
2510,2025-12-26,-0.0006,-0.0032,0.0009,0.0002
2511,2025-12-29,-0.0041,-0.0018,0.0007,0.0002
2512,2025-12-30,-0.0020,-0.0060,0.0028,0.0002
2513,2025-12-31,-0.0076,0.0007,-0.0009,0.0002


In [18]:
# =============================================================================
# 2D. FRED MACRO DATA (free — requires API key)
# =============================================================================

def load_fred_macro(start_date=TRAIN_START, end_date=DATA_END, api_key=None):
    """
    Load VIX, term spread (10Y-2Y), and 3-month T-bill from FRED.
    Falls back to yfinance for VIX if fredapi isn't available.

    Get a free API key at: https://fred.stlouisfed.org/docs/api/api_key.html

    Returns: DataFrame with date, vix, term_spread, tbill_3m
    """
    if api_key is None:
        api_key = os.environ.get("FRED_API_KEY")

    try:
        from fredapi import Fred

        if api_key is None:
            raise ValueError("No FRED API key. Set FRED_API_KEY env variable.")

        fred = Fred(api_key=api_key)
        print("[FRED] Downloading macro indicators...")

        vix         = fred.get_series("VIXCLS", start_date, end_date)
        term_spread = fred.get_series("T10Y2Y", start_date, end_date)
        tbill_3m    = fred.get_series("DTB3",   start_date, end_date)

        macro = pd.DataFrame({
            "vix": vix,
            "term_spread": term_spread,
            "tbill_3m": tbill_3m,
        })
        macro.index.name = "date"
        macro = macro.reset_index()
        macro["date"] = pd.to_datetime(macro["date"])
        macro = macro.sort_values("date").ffill()  # fill weekends/holidays

    except (ImportError, ValueError) as e:
        print(f"[FRED] fredapi unavailable ({e}). Falling back to yfinance for VIX...")
        import yfinance as yf
        vix_data = yf.download("^VIX", start=start_date, end=end_date, progress=False)
        macro = pd.DataFrame({
            "date": vix_data.index,
            "vix": vix_data["Close"].values.flatten(),
            "term_spread": np.nan,
            "tbill_3m": np.nan,
        })
        macro["date"] = pd.to_datetime(macro["date"])

    print(f"[FRED] ✓ {len(macro):,} daily macro observations")
    return macro

# --- Run it ---
# Set your FRED API key here, or use the yfinance fallback
# os.environ["FRED_API_KEY"] = "your_key_here"
fred_df = load_fred_macro()
fred_df.tail()

[FRED] fredapi unavailable (No FRED API key. Set FRED_API_KEY env variable.). Falling back to yfinance for VIX...
[FRED] ✓ 2,513 daily macro observations


,date,vix,term_spread,tbill_3m
2508,2025-12-23,14.0000,NaN,NaN
2509,2025-12-24,13.4700,NaN,NaN
2510,2025-12-26,13.6000,NaN,NaN
2511,2025-12-29,14.2000,NaN,NaN
2512,2025-12-30,14.3300,NaN,NaN


---
## Section 3 — Merge All Data Sources

Combine CRSP (daily), Compustat (quarterly, forward-filled), Fama-French (daily), and FRED (daily) into a single DataFrame with one row per (stock, date).

In [19]:
compustat_df["permno"] = compustat_df["permno"].astype(int)
crsp_df["permno"] = crsp_df["permno"].astype(int)

In [20]:
# =============================================================================
# 3. MERGE ALL SOURCES
# =============================================================================

def merge_all_sources(crsp, compustat, ff, fred):
    """
    Merge all data sources into one DataFrame.

    - Compustat: as-of merge (forward-fills quarterly data to daily, respecting lag)
    - FF and FRED: simple date join
    """
    print("[MERGE] Combining all sources...")

    # 3A. Compustat → CRSP (as-of merge using available_date to respect lag)
    if not compustat.empty:
        df = pd.merge_asof(
            crsp.sort_values("date"),
            compustat[["permno", "available_date", "pe_ratio"]]
                .sort_values("available_date")
                .rename(columns={"available_date": "date"}),
            on="date",
            by="permno",
            direction="backward",
        )
    else:
        df = crsp.copy()
        df["pe_ratio"] = np.nan

    # 3B. Fama-French → merge on date
    df = pd.merge(df, ff, on="date", how="left")

    # 3C. FRED macro → merge on date
    df = pd.merge(df, fred, on="date", how="left")

    # Forward-fill any remaining gaps (weekends, holidays)
    fill_cols = ["mkt_rf", "smb", "hml", "rf", "vix", "term_spread", "tbill_3m"]
    for col in fill_cols:
        if col in df.columns:
            df[col] = df.groupby("permno")[col].transform(lambda x: x.ffill())

    df = df.sort_values(["permno", "date"]).reset_index(drop=True)

    print(f"[MERGE] ✓ {len(df):,} rows, {df.columns.size} columns")
    print(f"[MERGE]   Stocks: {df['permno'].nunique()}, "
          f"Dates: {df['date'].min().date()} → {df['date'].max().date()}")
    return df

# --- Run it ---
merged_df = merge_all_sources(crsp_df, compustat_df, ff_df, fred_df)
print(f"\nColumns: {list(merged_df.columns)}")
merged_df.head()

[MERGE] Combining all sources...
[MERGE] ✓ 46,865 rows, 17 columns
[MERGE]   Stocks: 22, Dates: 2016-01-04 → 2024-12-31

Columns: ['permno', 'date', 'ret', 'prc', 'vol', 'shrout', 'mkt_cap', 'ticker', 'sector', 'pe_ratio', 'mkt_rf', 'smb', 'hml', 'rf', 'vix', 'term_spread', 'tbill_3m']


,permno,date,ret,prc,vol,shrout,mkt_cap,ticker,sector,pe_ratio,mkt_rf,smb,hml,rf,vix,term_spread,tbill_3m
0,10104,2016-01-04,-0.0172,35.7500,18784386.0000,4201220.0000,150193615.0000,ORCL,Information Technology,NaN,-0.0159,-0.0087,0.0052,0.0000,20.7000,NaN,NaN
1,10104,2016-01-05,-0.0031,35.6400,25340678.0000,4201220.0000,149731480.8000,ORCL,Information Technology,NaN,0.0013,-0.0018,0.0000,0.0000,19.3400,NaN,NaN
2,10104,2016-01-06,0.0051,35.8200,18165714.0000,4201220.0000,150487700.4000,ORCL,Information Technology,NaN,-0.0134,-0.0013,0.0001,0.0000,20.5900,NaN,NaN
3,10104,2016-01-07,-0.0218,35.0400,22591403.0000,4201220.0000,147210748.8000,ORCL,Information Technology,NaN,-0.0244,-0.0029,0.0008,0.0000,24.9900,NaN,NaN
4,10104,2016-01-08,-0.0111,34.6500,21962204.0000,4201220.0000,145572273.0000,ORCL,Information Technology,NaN,-0.0111,-0.0049,-0.0003,0.0000,27.0100,NaN,NaN


---
## Section 4 — Feature Engineering

This is the most important section. It transforms raw data into the ML-ready features listed on the chalkboard:

**From CRSP (per stock, per day):**
- Lagged returns (1d, 5d, 21d)
- Rolling volatility (21d, 63d)
- Momentum (3m, 6m, 12m)
- Moving average ratios (price / SMA-20, SMA-50, SMA-200)
- Volume ratio (5d avg / 63d avg)
- Log market cap

**From Compustat:**
- P/E ratio (trailing 12-month, lagged 60+ days)

**From Fama-French:**
- Rolling beta to market
- Rolling SMB and HML factor loadings

**From FRED:**
- VIX level, term spread

**Target variable:**
- Forward 21-day cumulative return

In [21]:
# =============================================================================
# 4. FEATURE ENGINEERING
# =============================================================================

def engineer_features(df, include_target=True):
    """
    Compute all ML features and (optionally) the forward-return target.

    Parameters
    ----------
    df : pd.DataFrame — output of merge_all_sources()
    include_target : bool — if False, skip forward return (for live inference)

    Returns
    -------
    pd.DataFrame with all original columns + engineered features + target
    """
    print("[FEATURES] Engineering features...")
    df = df.copy().sort_values(["permno", "date"]).reset_index(drop=True)
    g = df.groupby("permno")

    # ---- Lagged Returns ----
    df["ret_lag_1d"] = g["ret"].shift(1)

    df["ret_lag_5d"] = g["ret"].transform(
        lambda x: (1 + x).rolling(5).apply(np.prod, raw=True) - 1
    ).shift(1)

    df["ret_lag_21d"] = g["ret"].transform(
        lambda x: (1 + x).rolling(21).apply(np.prod, raw=True) - 1
    ).shift(1)

    # ---- Rolling Volatility ----
    df["vol_21d"] = g["ret"].transform(lambda x: x.rolling(21, min_periods=15).std())
    df["vol_63d"] = g["ret"].transform(lambda x: x.rolling(63, min_periods=40).std())

    # ---- Momentum ----
    for label, window in [("mom_3m", 63), ("mom_6m", 126), ("mom_12m", 252)]:
        min_p = int(window * 0.7)
        df[label] = g["ret"].transform(
            lambda x: (1 + x).rolling(window, min_periods=min_p).apply(np.prod, raw=True) - 1
        )

    # ---- Moving Averages & Price Ratios ----
    for n in [20, 50, 200]:
        sma_col = f"sma_{n}"
        df[sma_col] = g["prc"].transform(lambda x: x.rolling(n, min_periods=int(n * 0.7)).mean())
        df[f"prc_to_{sma_col}"] = df["prc"] / df[sma_col]

    # ---- Volume Ratio ----
    vol_5  = g["vol"].transform(lambda x: x.rolling(5, min_periods=3).mean())
    vol_63 = g["vol"].transform(lambda x: x.rolling(63, min_periods=40).mean())
    df["vol_ratio"] = vol_5 / vol_63.replace(0, np.nan)

    # ---- Log Market Cap ----
    df["log_mkt_cap"] = np.log(df["mkt_cap"].clip(lower=1))

    # ---- Rolling Beta (252-day) ----
    df["excess_ret"] = df["ret"] - df["rf"]

    def rolling_beta(group):
        excess = group["excess_ret"]
        mkt = group["mkt_rf"]
        cov = excess.rolling(252, min_periods=126).cov(mkt)
        var = mkt.rolling(252, min_periods=126).var()
        return cov / var.replace(0, np.nan)

    df["rolling_beta"] = g.apply(rolling_beta, include_groups=False).reset_index(level=0, drop=True)

    # ---- Rolling Factor Loadings (SMB, HML) ----
    for factor in ["smb", "hml"]:
        def _loading(group, f=factor):
            cov = group["excess_ret"].rolling(252, min_periods=126).cov(group[f])
            var = group[f].rolling(252, min_periods=126).var()
            return cov / var.replace(0, np.nan)

        df[f"rolling_{factor}_loading"] = (
            g.apply(_loading, include_groups=False).reset_index(level=0, drop=True)
        )

    df = df.drop(columns=["excess_ret"])

    # ---- Winsorize P/E ----
    if "pe_ratio" in df.columns and df["pe_ratio"].notna().any():
        lo, hi = df["pe_ratio"].quantile(0.01), df["pe_ratio"].quantile(0.99)
        df["pe_ratio"] = df["pe_ratio"].clip(lo, hi)

    # ---- Target Variable ----
    if include_target:
        df["fwd_21d_ret"] = g["ret"].transform(
            lambda x: (1 + x).shift(-FORWARD_RETURN_HORIZON)
            .rolling(FORWARD_RETURN_HORIZON).apply(np.prod, raw=True) - 1
        )

    # ---- Store feature column names for downstream use ----
    feature_cols = [
        "ret_lag_1d", "ret_lag_5d", "ret_lag_21d",
        "vol_21d", "vol_63d",
        "mom_3m", "mom_6m", "mom_12m",
        "prc_to_sma_20", "prc_to_sma_50", "prc_to_sma_200",
        "vol_ratio", "log_mkt_cap",
        "pe_ratio",
        "rolling_beta", "rolling_smb_loading", "rolling_hml_loading",
        "vix", "term_spread",
    ]
    df.attrs["feature_columns"] = [c for c in feature_cols if c in df.columns]
    df.attrs["target_column"] = "fwd_21d_ret" if include_target else None

    n_complete = df.dropna(subset=[c for c in feature_cols if c in df.columns]).shape[0]
    print(f"[FEATURES] ✓ {len(df.attrs['feature_columns'])} features engineered")
    print(f"[FEATURES]   Complete rows: {n_complete:,} / {len(df):,} ({n_complete/len(df):.1%})")
    return df

# --- Run it ---
featured_df = engineer_features(merged_df, include_target=True)
print(f"\nFeature columns: {featured_df.attrs['feature_columns']}")
featured_df.head()

[FEATURES] Engineering features...
[FEATURES] ✓ 19 features engineered
[FEATURES]   Complete rows: 0 / 46,865 (0.0%)

Feature columns: ['ret_lag_1d', 'ret_lag_5d', 'ret_lag_21d', 'vol_21d', 'vol_63d', 'mom_3m', 'mom_6m', 'mom_12m', 'prc_to_sma_20', 'prc_to_sma_50', 'prc_to_sma_200', 'vol_ratio', 'log_mkt_cap', 'pe_ratio', 'rolling_beta', 'rolling_smb_loading', 'rolling_hml_loading', 'vix', 'term_spread']


,permno,date,ret,prc,vol,shrout,mkt_cap,ticker,sector,pe_ratio,mkt_rf,smb,hml,rf,vix,term_spread,tbill_3m,ret_lag_1d,ret_lag_5d,ret_lag_21d,vol_21d,vol_63d,mom_3m,mom_6m,mom_12m,sma_20,prc_to_sma_20,sma_50,prc_to_sma_50,sma_200,prc_to_sma_200,vol_ratio,log_mkt_cap,rolling_beta,rolling_smb_loading,rolling_hml_loading,fwd_21d_ret
0,10104,2016-01-04,-0.0172,35.7500,18784386.0000,4201220.0000,150193615.0000,ORCL,Information Technology,NaN,-0.0159,-0.0087,0.0052,0.0000,20.7000,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,<NA>,NaN,<NA>,NaN,18.8274,NaN,NaN,NaN,NaN
1,10104,2016-01-05,-0.0031,35.6400,25340678.0000,4201220.0000,149731480.8000,ORCL,Information Technology,NaN,0.0013,-0.0018,0.0000,0.0000,19.3400,NaN,NaN,-0.0172,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,<NA>,NaN,<NA>,NaN,18.8244,NaN,NaN,NaN,NaN
2,10104,2016-01-06,0.0051,35.8200,18165714.0000,4201220.0000,150487700.4000,ORCL,Information Technology,NaN,-0.0134,-0.0013,0.0001,0.0000,20.5900,NaN,NaN,-0.0031,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,<NA>,NaN,<NA>,NaN,18.8294,NaN,NaN,NaN,NaN
3,10104,2016-01-07,-0.0218,35.0400,22591403.0000,4201220.0000,147210748.8000,ORCL,Information Technology,NaN,-0.0244,-0.0029,0.0008,0.0000,24.9900,NaN,NaN,0.0051,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,<NA>,NaN,<NA>,NaN,18.8074,NaN,NaN,NaN,NaN
4,10104,2016-01-08,-0.0111,34.6500,21962204.0000,4201220.0000,145572273.0000,ORCL,Information Technology,NaN,-0.0111,-0.0049,-0.0003,0.0000,27.0100,NaN,NaN,-0.0218,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,<NA>,NaN,<NA>,NaN,18.7962,NaN,NaN,NaN,NaN


In [22]:
# Check which features are missing and how badly
for col in featured_df.attrs["feature_columns"]:
    pct_missing = featured_df[col].isna().mean() * 100
    print(f"  {col:<30s} {pct_missing:>6.1f}% missing")

  ret_lag_1d                        0.0% missing
  ret_lag_5d                        0.2% missing
  ret_lag_21d                       0.9% missing
  vol_21d                           0.7% missing
  vol_63d                           1.8% missing
  mom_3m                            2.0% missing
  mom_6m                            4.1% missing
  mom_12m                           8.0% missing
  prc_to_sma_20                     0.6% missing
  prc_to_sma_50                     1.6% missing
  prc_to_sma_200                    6.4% missing
  vol_ratio                         1.8% missing
  log_mkt_cap                       0.0% missing
  pe_ratio                         20.2% missing
  rolling_beta                      5.8% missing
  rolling_smb_loading               5.8% missing
  rolling_hml_loading               5.8% missing
  vix                               0.0% missing
  term_spread                     100.0% missing


In [23]:
# 1. Drop term_spread from the feature list (100% missing — no FRED API key)
featured_df.attrs["feature_columns"] = [
    c for c in featured_df.attrs["feature_columns"] if c != "term_spread"
]

# 2. Fill pe_ratio NaNs with per-stock median
featured_df["pe_ratio"] = featured_df.groupby("permno")["pe_ratio"].transform(
    lambda x: x.fillna(x.median())
)

# 3. Check completeness again
for col in featured_df.attrs["feature_columns"]:
    pct = featured_df[col].isna().mean() * 100
    print(f"  {col:<30s} {pct:>6.1f}% missing")

# 4. Check complete rows
feat_cols = featured_df.attrs["feature_columns"]
n_complete = featured_df.dropna(subset=feat_cols).shape[0]
print(f"\nComplete rows: {n_complete:,} / {len(featured_df):,} ({n_complete/len(featured_df):.1%})")

  ret_lag_1d                        0.0% missing
  ret_lag_5d                        0.2% missing
  ret_lag_21d                       0.9% missing
  vol_21d                           0.7% missing
  vol_63d                           1.8% missing
  mom_3m                            2.0% missing
  mom_6m                            4.1% missing
  mom_12m                           8.0% missing
  prc_to_sma_20                     0.6% missing
  prc_to_sma_50                     1.6% missing
  prc_to_sma_200                    6.4% missing
  vol_ratio                         1.8% missing
  log_mkt_cap                       0.0% missing
  pe_ratio                          4.0% missing
  rolling_beta                      5.8% missing
  rolling_smb_loading               5.8% missing
  rolling_hml_loading               5.8% missing
  vix                               0.0% missing

Complete rows: 41,497 / 46,865 (88.5%)


---
## Section 5 — Save the Dataset

Save the fully-featured DataFrame as a parquet file. This is the artifact that Members B and C work from.

In [26]:
# =============================================================================
# 5. SAVE TO DISK
# =============================================================================

SAVE_PATH = "/content/drive/MyDrive/870 Financial Analytics Project/features.parquet"

os.makedirs("/content/drive/MyDrive/870 Financial Analytics Project", exist_ok=True)
featured_df.to_parquet(SAVE_PATH, index=False)

file_size_mb = os.path.getsize(SAVE_PATH) / (1024 * 1024)
print(f"✅ Saved to {SAVE_PATH}")
print(f"   Shape: {featured_df.shape}")
print(f"   Size:  {file_size_mb:.1f} MB")
print(f"   Stocks: {featured_df['ticker'].nunique()}")
print(f"   Dates:  {featured_df['date'].min().date()} → {featured_df['date'].max().date()}")
print(f"\n   Share this file with Members B and C.")

✅ Saved to /content/drive/MyDrive/870 Financial Analytics Project/features.parquet
   Shape: (46865, 37)
   Size:  9.9 MB
   Stocks: 22
   Dates:  2016-01-04 → 2024-12-31

   Share this file with Members B and C.


---
## Section 6 — Black-Litterman Inputs

These functions compute the equilibrium prior (what the market "believes" each stock will return, implied by market-cap weights) and blend it with ML views. **Member C uses these directly.**

### How Black-Litterman works:
1. **Equilibrium returns (π):** Derived from market-cap weights → π = δ × Σ × w_mkt
2. **ML views (Q):** Your XGBoost/RF return predictions
3. **Blended returns (μ_BL):** Weighted combination of π and Q, where the weight depends on model confidence

In [27]:
# =============================================================================
# 6A. COMPUTE BLACK-LITTERMAN EQUILIBRIUM INPUTS
# =============================================================================

def compute_bl_inputs(df, as_of_date=None, trailing_days=252, delta=BL_RISK_AVERSION,
                      use_shrinkage=True):
    """
    Compute the Black-Litterman prior from historical data.

    Returns dict with: sigma, w_mkt, pi, rf, tickers, as_of_date
    """
    from sklearn.covariance import LedoitWolf

    if as_of_date is None:
        as_of_date = df["date"].max()
    as_of_date = pd.Timestamp(as_of_date)

    # Filter to trailing window
    cutoff = as_of_date - pd.Timedelta(days=int(trailing_days * 1.6))
    window = df[(df["date"] <= as_of_date) & (df["date"] > cutoff)].copy()

    # Pivot to (dates × stocks) return matrix
    ret_wide = window.pivot_table(index="date", columns="ticker", values="ret").dropna()
    ret_wide = ret_wide.tail(trailing_days)
    tickers = list(ret_wide.columns)
    n = len(tickers)

    print(f"[BL] Computing inputs as of {as_of_date.date()} "
          f"({len(ret_wide)} days, {n} stocks)")

    # Covariance matrix (annualized)
    if use_shrinkage:
        lw = LedoitWolf().fit(ret_wide.values)
        sigma = lw.covariance_ * 252
        print(f"[BL]   Ledoit-Wolf shrinkage: {lw.shrinkage_:.3f}")
    else:
        sigma = ret_wide.cov().values * 252

    # Market-cap weights
    latest = window[window["date"] == window["date"].max()].set_index("ticker")["mkt_cap"]
    latest = latest.reindex(tickers)
    w_mkt = (latest / latest.sum()).values

    # Risk-free rate (annualized)
    rf_daily = window["rf"].dropna().iloc[-1] if "rf" in window.columns else 0.0
    rf = rf_daily * 252

    # Equilibrium excess returns: π = δ × Σ × w_mkt
    pi = delta * sigma @ w_mkt

    return {
        "sigma": sigma, "w_mkt": w_mkt, "pi": pi,
        "rf": rf, "tickers": tickers, "as_of_date": str(as_of_date.date()),
    }

In [28]:
# =============================================================================
# 6B. APPLY BLACK-LITTERMAN (Blend ML Views with Equilibrium)
# =============================================================================

def apply_black_litterman(bl_inputs, views_q, views_p=None, views_omega=None,
                          tau=BL_TAU):
    """
    Blend ML views with market equilibrium using Black-Litterman.

    Parameters
    ----------
    bl_inputs   : dict from compute_bl_inputs()
    views_q     : array (K,) — ML-predicted expected returns (annualized)
    views_p     : array (K×N) — pick matrix (default: identity for absolute views)
    views_omega : array (K×K) — view uncertainty (default: proportional to variance)
    tau         : float — equilibrium uncertainty scaling

    Returns
    -------
    dict with: mu_bl (blended returns), sigma_bl (blended covariance), tickers
    """
    sigma = bl_inputs["sigma"]
    pi = bl_inputs["pi"]
    n = len(pi)

    if views_p is None:
        views_p = np.eye(n)
    if views_omega is None:
        views_omega = np.diag(np.diag(tau * views_p @ sigma @ views_p.T))

    # Black-Litterman master formula
    tau_sigma_inv = np.linalg.inv(tau * sigma)
    omega_inv = np.linalg.inv(views_omega)
    posterior_prec = tau_sigma_inv + views_p.T @ omega_inv @ views_p

    mu_bl = np.linalg.solve(
        posterior_prec,
        tau_sigma_inv @ pi + views_p.T @ omega_inv @ views_q,
    )
    sigma_bl = sigma + np.linalg.inv(posterior_prec)

    return {"mu_bl": mu_bl, "sigma_bl": sigma_bl, "tickers": bl_inputs["tickers"]}

In [29]:
# =============================================================================
# 6C. TEST BLACK-LITTERMAN INPUTS
# =============================================================================

bl = compute_bl_inputs(featured_df)

print(f"\nEquilibrium returns (annualized):")
for t, r, w in zip(bl["tickers"], bl["pi"], bl["w_mkt"]):
    print(f"  {t:6s}  π = {r:>7.2%}   weight = {w:>6.2%}")

print(f"\nRisk-free rate: {bl['rf']:.2%}")
print(f"Covariance matrix shape: {bl['sigma'].shape}")

# Sanity check: with zero-confidence views, BL output should match equilibrium
dummy_views = bl["pi"].copy()  # views = equilibrium (no new info)
result = apply_black_litterman(bl, dummy_views)
diff = np.max(np.abs(result["mu_bl"] - bl["pi"]))
print(f"\nSanity check (views = equilibrium): max diff = {diff:.2e} (should be ~0)")

[BL] Computing inputs as of 2024-12-31 (252 days, 19 stocks)
[BL]   Ledoit-Wolf shrinkage: 0.091

Equilibrium returns (annualized):
  AAPL    π =   5.98%   weight = 23.64%
  ABBV    π =   2.55%   weight =  3.41%
  AMZN    π =   7.43%   weight = 14.41%
  BAC     π =   3.09%   weight =  2.11%
  BRK-B   π =   2.17%   weight =  3.76%
  CVX     π =   1.58%   weight =  1.63%
  GS      π =   4.26%   weight =  1.12%
  HD      π =   2.57%   weight =  2.41%
  JNJ     π =  -0.12%   weight =  2.17%
  JPM     π =   3.09%   weight =  4.21%
  MCD     π =   1.48%   weight =  1.30%
  MRK     π =   0.90%   weight =  1.57%
  MSFT    π =   5.74%   weight = 19.57%
  NKE     π =   2.93%   weight =  0.56%
  ORCL    π =   5.86%   weight =  2.91%
  SLB     π =   2.02%   weight =  1.26%
  TSLA    π =  15.06%   weight =  8.10%
  UNH     π =   0.14%   weight =  2.91%
  XOM     π =   0.33%   weight =  2.95%

Risk-free rate: 5.04%
Covariance matrix shape: (19, 19)

Sanity check (views = equilibrium): max diff = 2.0

---
## Section 7 — Interfaces for Members B and C

### 7A: Member B (ML) — Getting Training Data

In [30]:
# =============================================================================
# 7A. MEMBER B: GET CLEAN TRAINING DATA
# =============================================================================

def get_clean_training_data(df):
    """
    Extract clean (X, y, meta) arrays for ML training.
    Drops any rows with missing features or missing target.
    """
    feature_cols = df.attrs.get("feature_columns", [])
    target_col = df.attrs.get("target_column", "fwd_21d_ret")

    required = feature_cols + [target_col]
    clean = df.dropna(subset=required).copy()

    X = clean[feature_cols]
    y = clean[target_col]
    meta = clean[["permno", "date", "ticker", "sector"]]

    print(f"[TRAINING DATA] ✓ {len(X):,} clean samples, {len(feature_cols)} features")
    print(f"  Date range: {meta['date'].min().date()} → {meta['date'].max().date()}")
    print(f"  Stocks: {meta['ticker'].nunique()}")
    return X, y, meta

# --- Example usage ---
X, y, meta = get_clean_training_data(featured_df)

print(f"\nX shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"\nFeatures: {list(X.columns)}")
print(f"\ny (forward 21d return) summary:")
print(y.describe())

[TRAINING DATA] ✓ 41,077 clean samples, 18 features
  Date range: 2016-09-13 → 2024-11-29
  Stocks: 20

X shape: (41077, 18)
y shape: (41077,)

Features: ['ret_lag_1d', 'ret_lag_5d', 'ret_lag_21d', 'vol_21d', 'vol_63d', 'mom_3m', 'mom_6m', 'mom_12m', 'prc_to_sma_20', 'prc_to_sma_50', 'prc_to_sma_200', 'vol_ratio', 'log_mkt_cap', 'pe_ratio', 'rolling_beta', 'rolling_smb_loading', 'rolling_hml_loading', 'vix']

y (forward 21d return) summary:
count   41077.0000
mean        0.0168
std         0.0850
min        -0.5792
25%        -0.0278
50%         0.0152
75%         0.0580
max         1.0871
Name: fwd_21d_ret, dtype: float64


### 7B: Member C (Optimization) — Getting BL Inputs

Member C has already seen `compute_bl_inputs()` and `apply_black_litterman()` in Section 6. Here's a complete example of the optimization step.

In [31]:
# =============================================================================
# 7B. MEMBER C: FULL OPTIMIZATION EXAMPLE
# =============================================================================

from scipy.optimize import minimize

# Get BL inputs
bl = compute_bl_inputs(featured_df)

# Simulate ML views (in production, these come from Member B's model)
# For now, just add small random noise to equilibrium returns
np.random.seed(42)
ml_views = bl["pi"] + np.random.normal(0, 0.02, size=len(bl["pi"]))

# Prediction confidence (lower = more uncertain)
prediction_variance = np.full(len(bl["pi"]), 0.04)
omega = np.diag(prediction_variance)

# Apply Black-Litterman
bl_result = apply_black_litterman(bl, views_q=ml_views, views_omega=omega)
mu_bl = bl_result["mu_bl"]
sigma_bl = bl_result["sigma_bl"]
rf = bl["rf"]

# --- Optimize: Maximize Sharpe Ratio ---
n = len(mu_bl)

def neg_sharpe(w):
    port_ret = w @ mu_bl
    port_vol = np.sqrt(w @ sigma_bl @ w)
    if port_vol < 1e-10:
        return 0.0
    return -(port_ret - rf) / port_vol

constraints = [{"type": "eq", "fun": lambda w: np.sum(w) - 1}]  # fully invested
bounds = [(0, 0.15)] * n   # no short selling, max 15% per stock

result = minimize(
    neg_sharpe,
    x0=np.ones(n) / n,    # start from equal weight
    method="SLSQP",
    bounds=bounds,
    constraints=constraints,
)

w_opt = result.x
sharpe = -result.fun

print(f"Optimal Sharpe Ratio: {sharpe:.4f}")
print(f"Portfolio Return:     {(w_opt @ mu_bl):.2%}")
print(f"Portfolio Volatility: {np.sqrt(w_opt @ sigma_bl @ w_opt):.2%}\n")

print("Optimal Weights:")
print(f"  {'Ticker':<8} {'Sector':<28} {'Weight':>8}  {'BL Return':>10}")
for t, w, r in sorted(zip(bl["tickers"], w_opt, mu_bl), key=lambda x: -x[1]):
    if w > 0.001:
        s = TICKER_TO_SECTOR.get(t, "")
        print(f"  {t:<8} {s:<28} {w:>7.2%}  {r:>9.2%}")

[BL] Computing inputs as of 2024-12-31 (252 days, 19 stocks)
[BL]   Ledoit-Wolf shrinkage: 0.091
Optimal Sharpe Ratio: 0.0919
Portfolio Return:     6.89%
Portfolio Volatility: 20.08%

Optimal Weights:
  Ticker   Sector                         Weight   BL Return
  AAPL     Information Technology        15.00%      5.93%
  AMZN     Consumer Discretionary        15.00%      7.43%
  GS       Financials                    15.00%      4.51%
  MSFT     Information Technology        15.00%      5.70%
  TSLA     Consumer Discretionary        15.00%     14.54%
  ORCL     Information Technology        15.00%      5.60%
  BAC      Financials                     5.03%      3.31%
  JPM      Financials                     4.97%      3.30%


---
## Section 8 — Live Data Pipeline (yfinance)

This section mirrors the CRSP-based feature engineering but pulls from yfinance. Use this for:
- **Live inference** in the Streamlit app
- **Development** by Members B and C (works without WRDS)

In [32]:
# =============================================================================
# 8. LIVE DATA FROM YFINANCE
# =============================================================================

def prepare_live_features(tickers=None, lookback_days=400):
    """
    Pull recent data from yfinance and compute features matching the CRSP pipeline.
    Returns one row per stock with the latest feature values.
    """
    import yfinance as yf

    if tickers is None:
        tickers = ALL_TICKERS

    print(f"[LIVE] Downloading {len(tickers)} stocks from yfinance...")
    end = datetime.today()
    start = end - timedelta(days=lookback_days)
    raw = yf.download(tickers, start=start, end=end, progress=False, group_by="ticker")

    if raw.empty:
        raise ValueError("yfinance returned no data.")

    # Reshape to long format matching CRSP structure
    records = []
    for ticker in tickers:
        try:
            t_data = raw[ticker].copy() if len(tickers) > 1 else raw.copy()
            t_data = t_data.dropna(subset=["Close"])
            t_data["ticker"] = ticker
            t_data["ret"] = t_data["Close"].pct_change()
            t_data["prc"] = t_data["Close"]
            t_data["vol"] = t_data["Volume"]
            t_data["sector"] = TICKER_TO_SECTOR.get(ticker, "Unknown")
            t_data["permno"] = TICKER_TO_PERMNO.get(ticker, -1)

            # Approximate market cap
            try:
                info = yf.Ticker(ticker).info
                shares = info.get("sharesOutstanding",
                                  info.get("impliedSharesOutstanding", np.nan))
                t_data["mkt_cap"] = t_data["prc"] * shares / 1000
            except Exception:
                t_data["mkt_cap"] = np.nan

            t_data.index.name = "date"
            t_data = t_data.reset_index()
            records.append(t_data[["date", "permno", "ticker", "sector",
                                   "ret", "prc", "vol", "mkt_cap"]])
        except Exception as e:
            print(f"[LIVE] ⚠️  Skipped {ticker}: {e}")

    live_df = pd.concat(records, ignore_index=True).sort_values(["permno", "date"])

    # Merge FF factors and FRED
    try:
        ff = load_ff_factors(start.strftime("%Y-%m-%d"), end.strftime("%Y-%m-%d"))
        live_df = pd.merge(live_df, ff, on="date", how="left")
    except Exception:
        for col in ["mkt_rf", "smb", "hml", "rf"]:
            live_df[col] = np.nan

    try:
        fred = load_fred_macro(start.strftime("%Y-%m-%d"), end.strftime("%Y-%m-%d"))
        live_df = pd.merge(live_df, fred, on="date", how="left")
    except Exception:
        for col in ["vix", "term_spread", "tbill_3m"]:
            live_df[col] = np.nan

    if "pe_ratio" not in live_df.columns:
        live_df["pe_ratio"] = np.nan

    # Engineer features (no forward target for live data)
    live_feat = engineer_features(live_df, include_target=False)

    # Return latest row per stock
    latest = live_feat.sort_values("date").groupby("ticker").tail(1).reset_index(drop=True)
    print(f"[LIVE] ✓ Live features ready for {len(latest)} stocks")
    return latest

In [33]:
# --- Test with 5 stocks (no WRDS needed) ---
live = prepare_live_features(["AAPL", "MSFT", "JPM", "XOM", "AMZN"])

feature_cols = live.attrs.get("feature_columns", [])
print(f"\nLive features ({len(feature_cols)} cols) for {len(live)} stocks:\n")
display_cols = ["ticker"] + [c for c in feature_cols if c in live.columns]
print(live[display_cols].to_string(index=False))

[LIVE] Downloading 5 stocks from yfinance...
[FF] Downloading via pandas_datareader...
[FF] ✓ 242 daily observations, 2025-03-13 to 2026-02-27
[FRED] fredapi unavailable (No FRED API key. Set FRED_API_KEY env variable.). Falling back to yfinance for VIX...
[FRED] ✓ 275 daily macro observations
[FEATURES] Engineering features...
[FEATURES] ✓ 19 features engineered
[FEATURES]   Complete rows: 0 / 1,375 (0.0%)
[LIVE] ✓ Live features ready for 5 stocks

Live features (19 cols) for 5 stocks:

ticker  ret_lag_1d  ret_lag_5d  ret_lag_21d  vol_21d  vol_63d  mom_3m  mom_6m  mom_12m  prc_to_sma_20  prc_to_sma_50  prc_to_sma_200  vol_ratio  log_mkt_cap  pe_ratio  rolling_beta  rolling_smb_loading  rolling_hml_loading     vix  term_spread
   XOM     -0.0015     -0.0462      -0.0523   0.0211   0.0181  0.1751  0.3744   0.5205         0.9497         0.9830          1.2203     0.7820      20.2639       NaN        0.1691               0.1608               0.7085 17.9400          NaN
  AAPL      0.0294 

---
## Cleanup

Close the WRDS connection when done.

In [34]:
# Close WRDS connection
try:
    conn.close()
    print("WRDS connection closed ✓")
except:
    print("No active connection to close")

WRDS connection closed ✓


---
## Summary

| What was built | Where it lives |
|----------------|---------------|
| CRSP daily data loader | `load_crsp_daily()` |
| Compustat P/E via CCM | `load_compustat_quarterly()` |
| Fama-French factors | `load_ff_factors()` |
| FRED macro indicators | `load_fred_macro()` |
| Merge all sources | `merge_all_sources()` |
| 19 engineered features | `engineer_features()` |
| Forward 21d return target | `engineer_features(include_target=True)` |
| Black-Litterman prior | `compute_bl_inputs()` |
| BL view blending | `apply_black_litterman()` |
| Live yfinance pipeline | `prepare_live_features()` |
| Clean ML training arrays | `get_clean_training_data()` |

**Saved artifact:** `data/features.parquet` — share this with Members B and C.